**EJEMPLO SECCIÓN 2.4. (FNN 784 -> 256 -> 10)**


In [ ]:
!pip install pytorch_lightning --quiet
!pip install ISLP --quiet
!pip install torchinfo --quiet

import numpy as np, pandas as pd
from matplotlib.pyplot import subplots
from sklearn.linear_model import \
(LinearRegression ,
LogisticRegression ,
Lasso)
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from ISLP import load_data
from ISLP.models import ModelSpec as MS
from sklearn.model_selection import \
(train_test_split ,
GridSearchCV)

import torch
from torch import nn
from torch.optim import RMSprop
from torch.utils.data import TensorDataset

from torchmetrics import (MeanAbsoluteError ,
R2Score)
from torchinfo import summary
from torchvision.io import read_image

from pytorch_lightning import Trainer
from pytorch_lightning.loggers import CSVLogger

#SEMILLA PARA QUE SIEMPRE DE EL MISMO RESULTADO (SEMILLA = 0 EN ESTE CASO)
from pytorch_lightning import seed_everything
seed_everything(0, workers=True)
torch.use_deterministic_algorithms(True, warn_only=True)

from torchvision.datasets import MNIST
from torchvision.models import (resnet50 ,
ResNet50_Weights)
from torchvision.transforms import (Resize ,
Normalize ,
CenterCrop ,
ToTensor)

from ISLP.torch import rec_num_workers
from ISLP.torch import SimpleDataModule
from ISLP.torch import SimpleModule
from ISLP.torch import ErrorTracker

In [ ]:
from ISLP.torch import rec_num_workers

max_num_workers = rec_num_workers()


(mnist_train ,
mnist_test) = [MNIST(root='data',
train=train ,
download=True ,
transform=ToTensor())
for train in [True , False]]
mnist_train

In [ ]:
from ISLP.torch import SimpleDataModule

mnist_dm = SimpleDataModule(mnist_train ,
mnist_test ,
validation=0.2,
num_workers=max_num_workers ,
batch_size =256)

for idx , (X_ ,Y_) in enumerate(mnist_dm.train_dataloader()):
    print('X: ', X_.shape)
    print('Y: ', Y_.shape)
    if idx >= 1:
        break

class MNISTModel(nn.Module):
  def __init__(self):
    super(MNISTModel , self).__init__()
    self.layer1 = nn.Sequential(
      nn.Flatten(),
      nn.Linear(28*28, 256),
      nn.ReLU())
    self._forward = nn.Sequential(
      self.layer1 ,
      nn.Linear(256, 10))

  def forward(self , x):
    return self._forward(x)

In [ ]:
mnist_model = MNISTModel()

summary(mnist_model ,
        input_data=X_,
        col_names=['input_size',
                    'output_size',
                    'num_params'])

In [ ]:
mnist_module = SimpleModule.classification(mnist_model, num_classes=10)
mnist_logger = CSVLogger('logs', name='MNIST')

mnist_trainer = Trainer(deterministic=True ,
                        max_epochs=30,
                        logger=mnist_logger ,
                        callbacks=[ErrorTracker()])
mnist_trainer.fit(mnist_module ,
                  datamodule=mnist_dm)

In [ ]:
def summary_plot(results ,
                  ax,
                  col='loss',
                  valid_legend='Tasa de acierto en validación',
                  training_legend='Tasa de acierto en entrenamiento',
                  ylabel='Loss',
                  fontsize=20):
    for (column ,
          color ,
          label) in zip([f'train_{col}_epoch',
                        f'valid_{col}'],
                        ['blue',
                        'orange'],
                        [training_legend ,
                        valid_legend]):
        results.dropna(subset=[column]).plot(x='epoch',
                                              y=column ,
                                              label=label ,
                                              marker="none",
                                              color=color ,
                                              ax=ax)
    ax.set_xlabel('Épocas')
    ax.set_ylabel(ylabel)
    return ax

In [ ]:
mnist_results = pd.read_csv(mnist_logger.experiment.
  metrics_file_path)
fig , ax = subplots(1, 1, figsize=(6, 6))
summary_plot(mnist_results ,
              ax,
              col='accuracy',
              ylabel='AccTaauracy')
ax.set_ylim([0.8, 1])
ax.set_ylabel('Tasa de acierto')
ax.set_title("MNIST FNN con una única capa oculta - Curvas de aprendizaje")
ax.set_xticks(np.linspace(0, 30, 7).astype(int));

mnist_trainer.test(mnist_module ,
datamodule=mnist_dm)


*   Con con seed = 0 da una tasa de precisión de 97,29% (Sin dropout)





**RED NEURONAL CONVOLUCIONAL MNIST (EJEMPLO SECCIÓN 3.7)**

In [ ]:
!pip install pytorch_lightning --quiet
!pip install ISLP --quiet
!pip install torchinfo --quiet

import numpy as np, pandas as pd
from matplotlib.pyplot import subplots
from sklearn.linear_model import \
(LinearRegression ,
LogisticRegression ,
Lasso)
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from ISLP import load_data
from ISLP.models import ModelSpec as MS
from sklearn.model_selection import \
(train_test_split ,
GridSearchCV)

import torch
from torch import nn
from torch.optim import RMSprop
from torch.utils.data import TensorDataset

from torchmetrics import (MeanAbsoluteError ,
R2Score)
from torchinfo import summary
from torchvision.io import read_image

from pytorch_lightning import Trainer
from pytorch_lightning.loggers import CSVLogger

#SEMILLA PARA QUE SIEMPRE DE EL MISMO RESULTADO (SEMILLA = 0 EN ESTE CASO)
from pytorch_lightning import seed_everything
seed_everything(0, workers=True)
torch.use_deterministic_algorithms(True, warn_only=True)

from torchvision.datasets import MNIST
from torchvision.models import (resnet50 ,
ResNet50_Weights)
from torchvision.transforms import (Resize ,
Normalize ,
CenterCrop ,
ToTensor)

from ISLP.torch import rec_num_workers
from ISLP.torch import SimpleDataModule
from ISLP.torch import SimpleModule
from ISLP.torch import ErrorTracker

In [ ]:
from ISLP.torch import rec_num_workers

max_num_workers = rec_num_workers()


(mnist_train ,
mnist_test) = [MNIST(root='data',
                    train=train ,
                    download=True ,
                    transform=ToTensor())
              for train in [True , False]]
mnist_train

In [ ]:
transform = ToTensor()

#Utilizo esto para convertir los datos de MNIST a tensores
mnist_train_X = torch.stack([transform(x) for x in mnist_train.data.numpy()])
mnist_test_X  = torch.stack([transform(x) for x in mnist_test.data.numpy()])

mnist_train = TensorDataset(mnist_train_X,
                            torch.tensor(mnist_train.targets))
mnist_test  = TensorDataset(mnist_test_X,
                            torch.tensor(mnist_test.targets))

In [ ]:
#Sirve para visualizar algunas imágenes del MNIST
fig, axes = subplots(5, 5, figsize=(6, 6))
rng = np.random.default_rng(4)
indices = rng.choice(np.arange(len(mnist_train)), 25,
                     replace=False).reshape((5, 5))
for i in range(5):
    for j in range(5):
        idx = indices[i, j]
        # mnist_train_X tiene forma (N, 1, 28, 28); utilizo squeeze para eliminar el canal ya que son imágenes en escala de grises
        axes[i, j].imshow(mnist_train_X[idx].squeeze(), cmap='gray',
                          interpolation=None)
        axes[i, j].set_xticks([])
        axes[i, j].set_yticks([])

In [ ]:
from ISLP.torch import rec_num_workers

max_num_workers = rec_num_workers()

mnist_dm = SimpleDataModule(mnist_train ,
                            mnist_test ,
                            validation=0.2,
                            num_workers=max_num_workers ,
                            batch_size =128)

In [ ]:
for idx , (X_ ,Y_) in enumerate(mnist_dm.train_dataloader()):
    print('X: ', X_.shape)
    print('Y: ', Y_.shape)
    if idx >= 1:
        break

In [ ]:
class BuildingBlock(nn.Module):
    def __init__(self,
                 in_channels,
                 out_channels):
        super(BuildingBlock, self).__init__()
        self.conv = nn.Conv2d(in_channels=in_channels,
                              out_channels=out_channels,
                              kernel_size=(3, 3),
                              padding='same')
        self.activation = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=(2, 2))

    def forward(self, x):
        return self.pool(self.activation(self.conv(x)))

In [ ]:
class MNIST_CNN(nn.Module):
    def __init__(self):
        super(MNIST_CNN, self).__init__()
        sizes = [(1, 32),
                 (32, 64)]
        self.conv = nn.Sequential(*[BuildingBlock(in_, out_)
                                    for in_, out_ in sizes])
        self.output = nn.Sequential(nn.Linear(64 * 7 * 7, 256),
                                    nn.ReLU(),
                                    nn.Linear(256, 10))

    def forward(self, x):
        val = self.conv(x)
        val = torch.flatten(val, start_dim=1)
        return self.output(val)

In [ ]:
mnist_cnn = MNIST_CNN()
summary(mnist_cnn,
        input_data=X_,
        col_names=['input_size',
                   'output_size',
                   'num_params'])

In [ ]:
mnist_optimizer = RMSprop(mnist_cnn.parameters(), lr=0.001)
mnist_module    = SimpleModule.classification(mnist_cnn,
                                              optimizer=mnist_optimizer,
                                              num_classes=10)
mnist_logger    = CSVLogger('logs', name='MNIST')

mnist_trainer = Trainer(deterministic=True,
                        max_epochs=30,
                        logger=mnist_logger,
                        callbacks=[ErrorTracker()])
mnist_trainer.fit(mnist_module,
                  datamodule=mnist_dm)

In [ ]:
def summary_plot(results ,
                  ax,
                  col='loss',
                  valid_legend='Tasa de acierto en validación',
                  training_legend='Tasa de acierto en entrenamiento',
                  ylabel='Loss',
                  fontsize=20):
    for (column ,
          color ,
          label) in zip([f'train_{col}_epoch',
                        f'valid_{col}'],
                        ['blue',
                        'orange'],
                        [training_legend ,
                        valid_legend]):
        results.dropna(subset=[column]).plot(x='epoch',
                                              y=column ,
                                              label=label ,
                                              marker="none",
                                              color=color ,
                                              ax=ax)
    ax.set_xlabel('Épocas')
    ax.set_ylabel(ylabel)
    return ax

In [ ]:
mnist_trainer.test(mnist_module, datamodule=mnist_dm)

log_path = mnist_logger.experiment.metrics_file_path
mnist_results = pd.read_csv(log_path)

fig, ax = subplots(1, 1, figsize=(6, 6))
summary_plot(mnist_results,
             ax,
             col='accuracy',
             ylabel='Accuracy')
ax.set_xticks(np.linspace(0, 30, 7).astype(int))
ax.set_ylabel('Tasa de Acierto')
ax.set_ylim([0.8, 1])
ax.set_title('CNN MNIST – Curvas de aprendizaje')
ax.legend(loc='lower right')

Con seed=0 da una tasa de acierto del 99,34% con RSProp